# Read alerts with pandas dataframe ``ftransfer_lsst_2026-05-06_481298``
- author : Sylvie Dagoret-campagne
- creation date : 2026-05-06
- documentation : https://doc.lsst.fink-broker.org/services/data_transfer/#standard

## config

In [ ]:
!ls

In [ ]:
metadata_download_path = "arrow_schema_ftransfer_lsst_2026-05-06_481298.metadata"
alerts_download_path = "ftransfer_lsst_2026-05-06_481298"

## Metadata

In [ ]:
import pyarrow.parquet as pq

arrow_schema = pq.read_schema(metadata_download_path)

In [ ]:
# arrow_schema

## Alerts with pandas

In [ ]:
import pandas as pd

# dtype_backend="pyarrow" prevents wrong type inference
pdf = pd.read_parquet(alerts_download_path, dtype_backend="pyarrow")

In [ ]:
pdf

## nested_pandas

> pip install nested_pandas

In [ ]:
import nested_pandas as npd
import numpy as np
import pandas as pd

# Similar to Pandas, but always uses PyArrow types, so no data casting is happening
df = npd.read_parquet(alerts_download_path)
# Loop over previous DIA-source light curves and append current source:
for _idx, row in df.head(5).iterrows():
    # Python dict
    source = row["diaSource"]
    # Nested-Pandas automatically represents nested structure as a DataFrame
    prv_sources = row["prvDiaSources"]
    all_sources = pd.concat([prv_sources, pd.DataFrame([source])], ignore_index=True)
    # Get peak flux per band using Pandas' groupy
    max_flux_per_band = all_sources[["band", "psfFlux"]].groupby("band").max()
    # Round values and convert to a Python dict
    max_flux_dict = max_flux_per_band.round(2).to_dict()["psfFlux"]
    print(f"diaSourceId {row['diaSource']['diaSourceId']} --- Peak PSF Flux {max_flux_dict}")

In [ ]:
import nested_pandas as npd

df = npd.read_parquet(
    alerts_download_path,
    # Just load the source information to save time and memory
    columns=["diaSource"],
)
# Unnest the only diaSource column to get all its fields as top-level columns
# Uses standard Pandas `.struct` "accessor"
df = df["diaSource"].struct.explode()
# Collect all sources for a given diaObjectId into a new "nested" column "sources"
nf = npd.NestedFrame.from_flat(
    df,
    # New index column
    on="diaObjectId",
    # Columns to leave unnested, none in this case
    base_columns=[],
    name="sources",
)

In [ ]:
df

In [ ]:
nf

In [ ]:
nf.sources["psfFlux"]

In [ ]:
nf.sources["psfFluxErr"]

In [ ]:
# Keep sources with flux/error > 10
# "." is used as a separator between the nested column name and the sub-column (field)
nf = nf.query("sources.psfFlux / sources.psfFluxErr > 1")
# Convert fluxes to magnitudes
nf["sources.psfMag"] = 31.4 - 2.5 * np.log10(nf["sources.psfFlux"])
nf["sources.psfMagErr"] = 2.5 / np.log(10) * nf["sources.psfFluxErr"] / nf["sources.psfFlux"]

# Sort by date
nf = nf.sort_values("sources.midpointMjdTai")


# Get the largest magnitude slope between two consecutive sources
def max_mag_slope(row):
    # row is a Python dict, keys are sub-column names, values are numpy arrays
    diff_mag = np.diff(row["sources.psfMag"])
    diff_time = np.diff(row["sources.midpointMjdTai"])
    pairwise_magerr_sums = np.hypot(row["sources.psfMagErr"][:-1], row["sources.psfMagErr"][1:])
    relaxed_diff_mag = diff_mag - pairwise_magerr_sums
    slopes = relaxed_diff_mag / diff_time
    return np.max(slopes)


# Run user function over rows, r-band only
nf = nf.query("sources.band == 'r'").map_rows(
    max_mag_slope,
    # Column name for the output
    output_names="max_mag_slope_r",
    # Keep existing columns
    append_columns=True,
)
nf

## fink_client

In [ ]:
# from fink_client.avro_utils import AlertReader
# r = AlertReader(alerts_download_path)
# alerts = r.to_list(size=1)
## Each alert is a dictionary
# alerts[0]["diaObject"]["diaObjectId"]


In [ ]:
# import polars as pl
# pdf = pl.read_avro(alerts_download_path)